**Auteur(s)** : Cheikhou Akhmed KANE

**Description** : Mise en place `ilots.shp`

# Création du Fichier `ilots.shp`

## 1. Description du projet

Ce notebook a pour objectif de générer le fichier `ilots.shp`, une des couches d'information géographique requise par le module agricole de MAELIA.

En partant du parcellaire enrichi et des tables de synthèse, il assemble les attributs nécessaires pour lier chaque îlot à un exploitant et à un type de sol.

---
## 2. Objectifs

* Charger le parcellaire enrichi, la table des exploitations et la table de synthèse des sols.
* Joindre toutes ces informations pour attribuer à chaque îlot les bons identifiants `ID_EXPL`, `ID_SOL` et `ID_ZH`.
* Créer un identifiant unique pour chaque îlot (`ID_ILOT`).
* Ajouter les colonnes restantes requises par MAELIA avec des valeurs fixes (données sur l'irrigation, la pente, etc.).
* Sélectionner et ordonner les colonnes finales.
* Sauvegarder le GeoDataFrame final au format Shapefile.

---
## 3. Fichiers en Entrée et en Sortie

### 3.1. Fichiers en Entrée
* **Parcellaire Enrichi :** `data/sols/shapefiles/processed/parcellaire_enrichi.shp`
* **Liste des Exploitations :** `includes_sassemeV1/modeleAgricole/agriculteurs/exploitations.csv`
* **Données de Synthèse des Sols :** `data/sols/csv/processed/donnees_typesDeSol_enrichies.csv`

### 3.2. Fichier en Sortie
* **Shapefile des Îlots :** `includes_sassemeV1/modeleAgricole/ilots/dansZone/ilots.shp`
---

In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [2]:
# --- 1. CHEMINS ---
base_dir = Path.cwd().parent.resolve()

# Fichiers en entrée
input_parcellaire_path = base_dir / "data" / "sols" / "shapefiles" / "processed" / "parcellaire_enrichi.shp"
input_exploitations_path = base_dir / "includes_sassemeV1" / "modeleAgricole" / "agriculteurs" / "exploitations.csv"
input_synthese_path = base_dir / "data" / "sols" / "csv" / "processed" / "donnees_typesDeSol_enrichies.csv"

# Fichier en sortie
output_ilots_path = base_dir / "includes_sassemeV1" / "modeleAgricole" / "ilots" / "dansZone" / "ilots.shp"

In [3]:
# --- 2. CHARGEMENT DES DONNÉES ---
try:
    gdf_parcellaire = gpd.read_file(input_parcellaire_path)
    df_exploitations = pd.read_csv(input_exploitations_path, sep=';')
    df_synthese = pd.read_csv(input_synthese_path, sep=';')
    print("✅ Fichiers chargés avec succès.")
except Exception as e:
    print(f"🚨 ERREUR lors du chargement des fichiers : {e}")

✅ Fichiers chargés avec succès.


In [4]:
# --- 3. JOINTURES POUR ENRICHIR LE PARCELLAIRE ---
# 3a. Joindre pour obtenir ID_SOL et ID_ZH depuis la table de synthèse
# On sélectionne seulement les colonnes clés pour une jointure propre
gdf_ilots = gdf_parcellaire.merge(
    df_synthese[['ZONE_PEDO', 'ID_SOL', 'ID_ZH']],
    on='ZONE_PEDO',
    how='left'
)

# 3b. Joindre pour obtenir ID_EXPL depuis la table des exploitations
# Pour cela, on doit recréer temporairement la table de correspondance NOM_UTILIS -> ID_EXPL
df_map_expl = gdf_parcellaire[['NOM_UTILIS']].copy().drop_duplicates().reset_index(drop=True)
df_map_expl['ID_EXPL'] = 'SSM1-' + (df_map_expl.index + 1).astype(str).str.zfill(4)
gdf_ilots = gdf_ilots.merge(df_map_expl, on='NOM_UTILIS', how='left')
print("✅ Jointures terminées.")

✅ Jointures terminées.


In [5]:
# --- 4. CRÉATION DES COLONNES FINALES ---
# 4a. Créer ID_ILOT (identifiant unique pour chaque parcelle/îlot)
gdf_ilots.reset_index(inplace=True, drop=True)
gdf_ilots['ID_ILOT'] = gdf_ilots.index + 1

# 4b. Ajouter les colonnes avec des valeurs fixes
gdf_ilots['CARACT_IRR'] = 'N'
gdf_ilots['MATERIEL'] = 0  # Utiliser 0 pour les champs numériques nuls dans les shapefiles
gdf_ilots['LISTE_EQUIS'] = None
gdf_ilots['PENTE_MOY'] = 0
gdf_ilots['PENTE_SWAT'] = 0
gdf_ilots['EQU_0'] = None
gdf_ilots['EQU_1'] = None
gdf_ilots['EQU_2'] = None
print("-> Colonnes restantes créées.")

-> Colonnes restantes créées.


In [6]:
# --- 5. SÉLECTION ET ORDONNANCEMENT FINAL ---
colonnes_finales = [
    'ID_ILOT', 'ID_EXPL', 'ID_SOL', 'ID_ZH', 'CARACT_IRR', 'MATERIEL',
    'LISTE_EQUIS', 'PENTE_MOY', 'PENTE_SWAT', 'EQU_0', 'EQU_1', 'EQU_2',
    'geometry'
]
gdf_ilots_final = gdf_ilots[colonnes_finales]
print("-> Sélection et ordonnancement des colonnes finales terminés.")

-> Sélection et ordonnancement des colonnes finales terminés.


In [7]:
# --- 6. SAUVEGARDE ---
output_ilots_path.parent.mkdir(parents=True, exist_ok=True)
gdf_ilots_final.to_file(output_ilots_path, driver='ESRI Shapefile', encoding='utf-8')
print(f"\n✅ Fichier 'ilots.shp' sauvegardé dans :\n   {output_ilots_path}")


✅ Fichier 'ilots.shp' sauvegardé dans :
   C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\includes_sassemeV1\modeleAgricole\ilots\dansZone\ilots.shp


C:\Users\Cheikhou\AppData\Local\Temp\ipykernel_4568\3979545956.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_ilots_final.to_file(output_ilots_path, driver='ESRI Shapefile', encoding='utf-8')
C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\.venv\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LISTE_EQUIS' to 'LISTE_EQUI'
  ogr_write(


In [8]:
# --- 7. VÉRIFICATION ---
print(f"\nDimensions finales : {gdf_ilots_final.shape[0]} îlots, {gdf_ilots_final.shape[1]} colonnes.")
display(gdf_ilots_final.head())


Dimensions finales : 749 îlots, 13 colonnes.


,ID_ILOT,ID_EXPL,ID_SOL,ID_ZH,CARACT_IRR,MATERIEL,LISTE_EQUIS,PENTE_MOY,PENTE_SWAT,EQU_0,EQU_1,EQU_2,geometry
0,1,SSM1-0001,SSM1-sableux-dior_cc_avec_arbr,SSM1,N,0,None,0,0,None,None,None,"MULTIPOLYGON (((337434.305 1603598.415, 337435..."
1,2,SSM1-0001,SSM1-sableux-dior_cc_sans_arbr,SSM1,N,0,None,0,0,None,None,None,"POLYGON ((337438.043 1603638.866, 337438.614 1..."
2,3,SSM1-0001,SSM1-argileux-dekk_cb_avec_arbr,SSM1,N,0,None,0,0,None,None,None,"MULTIPOLYGON (((336159.144 1603477.128, 336160..."
3,4,SSM1-0001,SSM1-argileux-dekk_cb_sans_arbr,SSM1,N,0,None,0,0,None,None,None,"POLYGON ((336165.254 1603521.577, 336167.671 1..."
4,5,SSM1-0001,SSM1-sableux-dior_cb_avec_arbr,SSM1,N,0,None,0,0,None,None,None,"MULTIPOLYGON (((336335.234 1602955.393, 336334..."
